# TRAINING BERT BASE WITH CRF

## SETTING UP THE ENVIRONMENT

In [1]:
!pip install -q --no-deps evaluate seqeval huggingface_hub pytorch-crf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.0 MB/s eta 0:00:00


In [ ]:
from huggingface_hub import login

In [ ]:
import os

os.environ["HF_TOKEN"] = ""

Importing the libraries:

In [ ]:
import torch
import spacy
import evaluate

import numpy    as np
import torch.nn as nn

from torchcrf import CRF

from pprint   import pprint
from spacy    import displacy
from datasets import Value       , \
                     Sequence    , \
                     Features    , \
                     ClassLabel  , \
                     DatasetDict , \
                     load_dataset, \
                     concatenate_datasets

from transformers import Trainer                    , \
                         AutoModel                  , \
                         AutoConfig                 , \
                         AutoTokenizer              , \
                         TrainingArguments          , \
                         TokenClassificationPipeline, \
                         AutoModelForTokenClassification

from transformers.modeling_outputs import TokenClassifierOutput

2025-11-25 16:57:08.882087: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764089829.069037      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764089829.120033      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

Load the dataset:

In [ ]:
sources = {
    "wikiann_pt" : ("wikiann"            , "pt"),
    "lener_br"   : ("lfcc/portuguese_ner", None),
}

datasets = {name : load_dataset(repo, subset) \
                   if   subset
                   else load_dataset(repo)
            for name, (repo, subset) in sources.items()}
datasets

README.md: 0.00B [00:00, ?B/s]

pt/validation-00000-of-00001.parquet:   0%|          | 0.00/636k [00:00<?, ?B/s]

pt/test-00000-of-00001.parquet:   0%|          | 0.00/628k [00:00<?, ?B/s]

pt/train-00000-of-00001.parquet:   0%|          | 0.00/1.26M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

...

data/train-00000-of-00001.parquet:   0%|          | 0.00/266k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/67.4k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3716 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/930 [00:00<?, ? examples/s]

{'wikiann_pt': DatasetDict({
     validation: Dataset({
         features: ['tokens', 'ner_tags', 'langs', 'spans'],
         num_rows: 10000
     })
     test: Dataset({
         features: ['tokens', 'ner_tags', 'langs', 'spans'],
         num_rows: 10000
     })
     train: Dataset({
         features: ['tokens', 'ner_tags', 'langs', 'spans'],
         num_rows: 20000
     })
 }),
 'lener_br': DatasetDict({
     train: Dataset({
         features: ['tokens', 'ner_tags'],
         num_rows: 3716
     })
     test: Dataset({
         features: ['tokens', 'ner_tags'],
         num_rows: 930
     })
 })}

In [6]:
features   = datasets["wikiann_pt"]["train"   ].features
label_list = features["ner_tags"              ].feature.names

num_labels = len(label_list)

id2label = {i     : label for i, label in enumerate(label_list)}
label2id = {label : i     for i, label in enumerate(label_list)}

print("Labels:", ", ".join(label_list))

Labels: O, B-PER, I-PER, B-ORG, I-ORG, B-LOC, I-LOC


In [7]:
def map_label(original):
    o = original.upper()

    if "PESSOA" in o or \
       "PER"    in o:
        if o.startswith("I-"):
            return "I-PER"

        return "B-PER"

    if "ORG"         in o or \
       "ORGANIZACAO" in o or \
       "ORGANIZAÇÃO" in o:
        if o.startswith("I-"):
            return "I-ORG"

        return "B-ORG"

    if "LOC"   in o or \
       "LOCAL" in o:
        if o.startswith("I-"):
            return "I-LOC"

        return "B-LOC"

    return "O"


def normalize_dataset(ds):
    train_features = ds["train"].features

    def convert(example):
        original_tags = example["ner_tags"]

        if isinstance(train_features["ner_tags"].feature, ClassLabel):
            names           = train_features["ner_tags"].feature.names
            original_labels = [names[t] for t in original_tags]
        else:
            original_labels = original_tags

        new_tags = [label2id[map_label(lbl)] for lbl in original_labels]

        return {
            "tokens"   : example["tokens"],
            "ner_tags" : new_tags
        }

    new_splits = {}
    for split in ["train", "validation", "test"]:
        if split in ds:
            new_splits[split] = ds[split].map(
                convert,
                remove_columns=ds[split].column_names
            )

    features = Features({
        "tokens"   : Sequence(Value("string")),
        "ner_tags" : Sequence(Value("int64" ))
    })

    for split in new_splits:
        new_splits[split] = new_splits[split].cast(features)

    return DatasetDict(new_splits)


normalized = []
for name, ds in datasets.items():
    print()
    print(f"Normalizing {name}...")

    normalized.append(normalize_dataset(ds))


def concat(split):
    parts = [
        ds[split]
        for ds in normalized
        if split in ds
    ]

    return concatenate_datasets(parts)


print()
print("Concatenating...")

train = concat("train"     )
val   = concat("validation")
test  = concat("test"      )

dataset = DatasetDict({
    "train"      : train,
    "validation" : val  ,
    "test"       : test ,
})


print()
print("Dataset:")
print(dataset)


Normalizing wikiann_pt...


Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/20000 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/3716 [00:00<?, ? examples/s]

Map:   0%|          | 0/930 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/3716 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/930 [00:00<?, ? examples/s]


Concatenating...

Dataset:
DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 23716
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 10930
    })
})


Tokenize and align the dataset:

In [ ]:
model_name = "neuralmind/bert-base-portuguese-cased"

tokenizer  = AutoTokenizer.from_pretrained(model_name)

def tokenize_and_align(example):
    tokenized = tokenizer(
        example["tokens"],
        is_split_into_words=True,
        truncation         =True,
        padding   ="max_length",
        max_length=128,
    )

    word_ids = tokenized.word_ids()
    labels   = example["ner_tags"]

    aligned = []
    for w in word_ids:
        if w is None:
            aligned.append(0)
        else:
            aligned.append(labels[w])

    tokenized["labels"] = aligned

    return tokenized


tokenized_dataset = dataset.map(
    tokenize_and_align,
    batched=False,
    remove_columns=["tokens", "ner_tags"]
)

tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/23716 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10930 [00:00<?, ? examples/s]

In [ ]:
example = {
    "tokens"   : ["João", "comprou", "uma", "maçã", "vermelha"],
    "ner_tags" : [1, 0, 0, 2, 2]
}

tok = tokenize_and_align(example)

print("Original tokens:")
print(example ["tokens"])
print("Original NER tags:")
print(example ["ner_tags"])

print()

print("Subtokens:")
print(tokenizer.convert_ids_to_tokens(tok["input_ids"])[:10])
print("Word ids:"      )
print(tok.word_ids()[:10])
print("Aligned labels:")
print(tok ["labels"][:10])

Original tokens:
['João', 'comprou', 'uma', 'maçã', 'vermelha']
Original NER tags:
[1, 0, 0, 2, 2]

Subtokens:
['[CLS]', 'João', 'comprou', 'uma', 'maç', '##ã', 'vermelha', '[SEP]', '[PAD]', '[PAD]']
Word ids:
[None, 0, 1, 2, 3, 3, 4, None, None, None]
Aligned labels:
[0, 1, 0, 0, 2, 2, 2, 0, 0, 0]


Auxiliary functions:

In [ ]:
seqeval = evaluate.load("seqeval")

def decode_predictions(predictions, labels):
    preds = np.argmax(predictions, axis=-1)
    pred_strings  = []
    label_strings = []

    for pred_ids, label_ids in zip(preds, labels):
        p_str = []
        l_str = []
        for p, l in zip(pred_ids, label_ids):
            if l == -100:
                continue

            p_str.append(label_list[p])
            l_str.append(label_list[l])

        pred_strings .append(p_str)
        label_strings.append(l_str)

    return pred_strings, label_strings


def compute_metrics(eval_pred):
    preds, refs = decode_predictions(*eval_pred)
    results     = seqeval.compute(
        predictions=preds,
        references =refs ,
    )

    return {
        "precision" : results["overall_precision"],
        "recall"    : results["overall_recall"   ],
        "f1"        : results["overall_f1"       ],
        "accuracy"  : results["overall_accuracy" ],
    }

## BERT BASE WITH CRF

In [ ]:
config = AutoConfig.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label  =id2label  ,
    label2id  =label2id  ,
)


class BertCRFForTokenClassification(nn.Module):
    def __init__(self, pretrained, config):
        super().__init__()

        self.config = config

        self.bert       = AutoModel.from_pretrained(pretrained, config=config)
        self.dropout    = nn.Dropout(0.1)
        self.classifier = nn.Linear (self.bert.config.hidden_size,
                                     config.num_labels)

        self.crf = CRF(num_tags=config.num_labels,
                       batch_first=True)

    def forward(self,
                input_ids     =None,
                attention_mask=None,
                token_type_ids=None,
                labels        =None,
                **kwargs):
        outputs = self.bert(
            input_ids     =input_ids     ,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            return_dict=True
        )

        sequence_output = outputs.last_hidden_state
        sequence_output = self.dropout   (sequence_output)
        emissions       = self.classifier(sequence_output)

        loss = None

        if labels is not None:
            mask = labels != -100
            labels_for_crf = labels.clone()
            labels_for_crf[~mask] = 0

            loss = -self.crf(emissions     ,
                             labels_for_crf,
                             mask     =mask,
                             reduction='mean')

            return TokenClassifierOutput(loss  =loss     ,
                                         logits=emissions,
                                         hidden_states=outputs.hidden_states,
                                         attentions   =outputs.attentions   )

        if attention_mask is None:
            mask = torch.ones(emissions.size()[:2],
                              dtype =torch.bool   ,
                              device=emissions.device)
        else:
            mask = attention_mask.bool()

        best_paths = self.crf.decode(emissions, mask=mask)

        logits_like = torch.zeros(emissions.size(),
                                  device=emissions.device)
        for i, path in enumerate(best_paths):
            for j, tag in enumerate(path):
                logits_like[i, j, tag] = 1.0

        return TokenClassifierOutput(loss  =None       ,
                                     logits=logits_like,
                                     hidden_states=outputs.hidden_states,
                                     attentions   =outputs.attentions   )


model = BertCRFForTokenClassification(model_name, config)

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

In [ ]:
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters       : {total_params    :,}")
print(f"Trainable parameters   : {trainable_params:,}")
print(f"Non-trainable params   : {total_params - trainable_params:,}")

Total parameters       : 108,928,582
Trainable parameters   : 108,928,582
Non-trainable params   : 0


In [ ]:
training_args = TrainingArguments(
    output_dir   ="results",
    eval_strategy="epoch"  ,
    save_strategy="epoch"  ,
    learning_rate=2e-5     ,

    per_device_train_batch_size=32,
    per_device_eval_batch_size =32,

    num_train_epochs=5     ,
    weight_decay    =0.01  ,
    report_to       ="none",
)

trainer = Trainer(
    model=model        ,
    args =training_args,

    train_dataset=tokenized_dataset["train"     ],
    eval_dataset =tokenized_dataset["validation"],

    processing_class=tokenizer      ,
    compute_metrics =compute_metrics,
)

trainer.train()

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,8.752400,2.332970,0.867644,0.882785,0.875149,0.994663
2,2.267700,2.118576,0.878586,0.897657,0.888020,0.995328
3,1.226100,2.110232,0.895417,0.905308,0.900336,0.995721
4,1.038800,2.215469,0.905173,0.910037,0.907598,0.995900
5,0.688900,2.291924,0.904937,0.911627,0.908269,0.995983


TrainOutput(global_step=3710, training_loss=2.273847126510908, metrics={'train_runtime': 2174.1271, 'train_samples_per_second': 54.541, 'train_steps_per_second': 1.706, 'total_flos': 0.0, 'train_loss': 2.273847126510908, 'epoch': 5.0})

In [17]:
test_metrics = trainer.evaluate(tokenized_dataset["test"])

print ("Test:")
pprint(test_metrics)

Test:
{'epoch': 5.0,
 'eval_accuracy': 0.9961580798261666,
 'eval_f1': 0.91884099241804,
 'eval_loss': 2.163350820541382,
 'eval_precision': 0.9136374898240894,
 'eval_recall': 0.9241041062542512,
 'eval_runtime': 65.3489,
 'eval_samples_per_second': 167.256,
 'eval_steps_per_second': 5.233}


In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

device

device(type='cuda')

In [16]:
def predict_entities(text):
    encoding = tokenizer(
        text,
        return_offsets_mapping=True,
        return_tensors        ="pt"
    )

    encoding = {k : v.to(device)
                for k, v in encoding.items()}

    with torch.no_grad():
        out = model(**encoding)

    logits   = out.logits
    pred_ids = logits.argmax(-1)[0].tolist()
    labels   = [model.config.id2label[i]
                for i in pred_ids]

    offsets   = encoding["offset_mapping"][0].tolist()
    input_ids = encoding["input_ids"     ][0].tolist()
    tokens    = tokenizer.convert_ids_to_tokens(input_ids)

    clean = []
    for tok, lab, (start, end) in zip(tokens,
                                      labels,
                                      offsets):
        if start == 0 and end == 0:
            continue

        clean.append((tok, lab, start, end))

    ents = []
    ent  = None
    for tok, lab, start, end in clean:
        if lab.startswith("B-"):
            if ent:
                ents.append(ent)

            ent = {
                "start" : start,
                "end"   : end  ,
                "label" : lab[2:],
            }
        elif lab.startswith("I-") and ent is not None:
            ent["end"] = end
        else:
            if ent:
                ents.append(ent)
            ent = None

    if ent:
        ents.append(ent)

    return ents


def ner_displacy(text):
    ents = predict_entities(text)

    doc = {
        "text" : text,
        "ents" : []  ,
        "title ": None
    }

    for e in ents:
        doc["ents"].append({
            "start" : e["start"],
            "end"   : e["end"  ],
            "label" : e["label"]
        })

    options = {
        "colors": {
            "PER" : "linear-gradient(90deg, #999999, #cccccc)",
            "LOC" : "linear-gradient(90deg, #aa9cfc, #fc9ce7)",
            "ORG" : "linear-gradient(90deg, #ffcc70, #ff9a3c)",
        }
    }

    return doc, options


test_texts = [
    "João encontrou Maria ontem à noite."     ,
    "Estou indo para João Pessoa amanhã cedo.",
    "A Google lançou um novo modelo de IA."   ,
    "A Universidade Federal da Paraíba convidou Ana Beatriz para apresentar sua pesquisa em São Paulo."                 ,
    "O presidente da Microsoft Brasil, André Oliveira, visitou o escritório em Fortaleza para anunciar novas parcerias.",
    "Mariana trabalhou três anos na IBM, antes de se mudar para o Rio de Janeiro para atuar no BNDES."                  ,
    "Em 2024, Carlos Eduardo foi contratado pelo Banco do Brasil após concluir seu mestrado na USP, em São Paulo."      ,
    "A Meta divulgou um relatório em que Sheryl Sandberg mencionou iniciativas de segurança digital nas operações da empresa."                ,
    "Durante a reunião em Brasília, representantes da ONU e do Ministério da Justiça discutiram estratégias para reduzir crimes cibernéticos.",
    "Pedro Henrique trabalhou por cinco anos na Petrobras no Rio de Janeiro, até receber uma proposta da Amazon em Seattle."
]

for i, text in enumerate(test_texts, 1):
    docs, options = ner_displacy(text)
    displacy.render(docs, style="ent", options=options, manual=True, jupyter=True)